# 10 - PyHMMER pAgo annotation

This notebook annotates the active protein FASTA/SWeeP records with pAgo-relevant Pfam HMM profiles using `pyhmmer`. It intentionally stays separate from the SWeeP embedding notebook.


In [1]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path


In [2]:
# =============================================================================
# CELL 2 - Resolve project root
# =============================================================================

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 - Bind pipeline module entry points
# =============================================================================

import src.pago_pipeline.ncbi_fasta_snapshot as ncbi_fasta_snapshot_module
import src.pago_pipeline.pyhmmer_pago as pyhmmer_pago_module
import src.pago_pipeline.sweep_genes_snapshot as sweep_genes_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_fasta_snapshot_module = importlib.reload(ncbi_fasta_snapshot_module)
pyhmmer_pago_module = importlib.reload(pyhmmer_pago_module)
sweep_genes_snapshot_module = importlib.reload(sweep_genes_snapshot_module)

load_latest_fasta_snapshot = ncbi_fasta_snapshot_module.load_latest_fasta_snapshot
load_latest_sweep_genes_snapshot = (
    sweep_genes_snapshot_module.load_latest_sweep_genes_snapshot
)
build_pyhmmer_pago_annotations = pyhmmer_pago_module.build_pyhmmer_pago_annotations
DEFAULT_PAGO_PFAM_ACCESSIONS = pyhmmer_pago_module.DEFAULT_PAGO_PFAM_ACCESSIONS

print("PyHMMER pAgo logic is provided by src/pago_pipeline/pyhmmer_pago.py")


PyHMMER pAgo logic is provided by src/pago_pipeline/pyhmmer_pago.py


In [4]:
# =============================================================================
# CELL 4 - Define PyHMMER pAgo configuration
# =============================================================================

def resolve_snapshot_root_directory(*, primary_path, fallback_path):
    if primary_path.exists() or not fallback_path.exists():
        return primary_path
    return fallback_path

FASTA_SNAPSHOT_ROOT_DIRECTORY = resolve_snapshot_root_directory(
    primary_path=PROJECT_ROOT
    / "data"
    / "02-intermediate"
    / "ncbi"
    / "protein_fasta",
    fallback_path=PROJECT_ROOT / "data" / "02-intermediate" / "protein_fasta",
)
SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "03-features" / "sweep_genes"
)
HMMER_PAGO_OUTPUT_DIRECTORY = (
    PROJECT_ROOT / "data" / "03-features" / "hmmer_pago" / "latest"
)

AVAILABLE_CPUS = os.cpu_count() or 1
HMMER_PAGO_PROFILE_ACCESSIONS = list(DEFAULT_PAGO_PFAM_ACCESSIONS)
HMMER_PAGO_CPUS = min(8, max(1, AVAILABLE_CPUS - 1))
HMMER_PAGO_BIT_CUTOFFS = "trusted"
HMMER_PAGO_FORCE_PROFILE_DOWNLOAD = False

print(f"FASTA snapshot root directory: {FASTA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"SWeeP Genes snapshot root directory: {SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY}")
print(f"PyHMMER pAgo output directory: {HMMER_PAGO_OUTPUT_DIRECTORY}")
print(f"PyHMMER pAgo Pfam profiles: {HMMER_PAGO_PROFILE_ACCESSIONS}")
print(f"PyHMMER pAgo CPUs: {HMMER_PAGO_CPUS}")
print(f"PyHMMER pAgo bit cutoffs: {HMMER_PAGO_BIT_CUTOFFS}")


FASTA snapshot root directory: C:\Programming\Python\pAgo-project\data\02-intermediate\protein_fasta
SWeeP Genes snapshot root directory: C:\Programming\Python\pAgo-project\data\03-features\sweep_genes
PyHMMER pAgo output directory: C:\Programming\Python\pAgo-project\data\03-features\hmmer_pago\latest
PyHMMER pAgo Pfam profiles: ['PF16486', 'PF08699', 'PF02170', 'PF16488', 'PF16487', 'PF02171', 'PF18157', 'PF18155', 'PF18156', 'PF18154']
PyHMMER pAgo CPUs: 8
PyHMMER pAgo bit cutoffs: trusted


In [5]:
# =============================================================================
# CELL 5 - Load active FASTA and SWeeP Genes snapshots
# =============================================================================

fasta_snapshot_payload = load_latest_fasta_snapshot(
    snapshot_root_directory=FASTA_SNAPSHOT_ROOT_DIRECTORY,
)
sweep_genes_snapshot_payload = load_latest_sweep_genes_snapshot(
    snapshot_root_directory=SWEEP_GENES_SNAPSHOT_ROOT_DIRECTORY,
)

protein_fasta_snapshot_directory = fasta_snapshot_payload["snapshot_directory"]
protein_fasta_manifest_file_path = fasta_snapshot_payload["manifest_file_path"]
protein_fasta_manifest_payload = fasta_snapshot_payload["manifest"]
protein_fasta_file_path = fasta_snapshot_payload["fasta_file_path"]

sweep_genes_snapshot_directory = sweep_genes_snapshot_payload["snapshot_directory"]
sweep_genes_manifest_file_path = sweep_genes_snapshot_payload["manifest_file_path"]
sweep_genes_manifest_payload = sweep_genes_snapshot_payload["manifest"]
sweep_genes_embeddings_file_path = sweep_genes_snapshot_payload["embeddings_file_path"]
sweep_genes_sequence_metadata_file_path = sweep_genes_snapshot_payload[
    "sequence_metadata_file_path"
]
sweep_genes_embeddings = sweep_genes_snapshot_payload["embeddings"]
sweep_genes_sequence_metadata_dataframe = sweep_genes_snapshot_payload[
    "sequence_metadata"
]

print("Loaded active FASTA and SWeeP Genes snapshots.")
print(f"FASTA file path: {protein_fasta_file_path}")
print(f"SWeeP sequence metadata file path: {sweep_genes_sequence_metadata_file_path}")
print(f"Sequence count: {len(sweep_genes_sequence_metadata_dataframe)}")
print(f"Embeddings shape: {sweep_genes_embeddings.shape}")


Loaded active FASTA and SWeeP Genes snapshots.
FASTA file path: C:\Programming\Python\pAgo-project\data\02-intermediate\protein_fasta\latest\protein_sequences.fasta
SWeeP sequence metadata file path: C:\Programming\Python\pAgo-project\data\03-features\sweep_genes\latest\sequence_metadata.csv
Sequence count: 41345
Embeddings shape: (41345, 2800)


In [6]:
# =============================================================================
# CELL 6 - Run PyHMMER pAgo domain annotation
# =============================================================================

pyhmmer_pago_result = build_pyhmmer_pago_annotations(
    protein_fasta_file_path=protein_fasta_file_path,
    sequence_metadata=sweep_genes_sequence_metadata_dataframe,
    output_directory=HMMER_PAGO_OUTPUT_DIRECTORY,
    profile_accessions=HMMER_PAGO_PROFILE_ACCESSIONS,
    cpus=HMMER_PAGO_CPUS,
    bit_cutoffs=HMMER_PAGO_BIT_CUTOFFS,
    force_profile_download=HMMER_PAGO_FORCE_PROFILE_DOWNLOAD,
)

pyhmmer_pago_domain_hits_dataframe = pyhmmer_pago_result.domain_hits
pyhmmer_pago_sequence_summary_dataframe = pyhmmer_pago_result.sequence_summary
pyhmmer_pago_annotated_metadata_dataframe = pyhmmer_pago_result.annotated_metadata
pyhmmer_pago_manifest_payload = pyhmmer_pago_result.manifest
pyhmmer_pago_manifest_file_path = pyhmmer_pago_result.manifest_file_path
pyhmmer_pago_domain_hits_file_path = pyhmmer_pago_result.domain_hits_file_path
pyhmmer_pago_sequence_summary_file_path = pyhmmer_pago_result.sequence_summary_file_path
pyhmmer_pago_annotated_metadata_file_path = (
    pyhmmer_pago_result.annotated_metadata_file_path
)

pyhmmer_classic_piwi_candidate_count = int(
    pyhmmer_pago_annotated_metadata_dataframe[
        "pyhmmer_is_classic_pago_candidate"
    ].sum()
)
pyhmmer_ppiwi_re_candidate_count = int(
    pyhmmer_pago_annotated_metadata_dataframe[
        "pyhmmer_is_ppiwi_re_candidate"
    ].sum()
)
pyhmmer_ppiwi_re_accessory_only_count = int(
    pyhmmer_pago_annotated_metadata_dataframe[
        "pyhmmer_is_ppiwi_re_accessory_candidate"
    ].sum()
)
pyhmmer_isolated_non_piwi_classic_count = int(
    (
        pyhmmer_pago_annotated_metadata_dataframe["pyhmmer_evidence_class"]
        == "isolated_non_piwi_classic_ago_profile"
    ).sum()
)
pyhmmer_no_selected_profile_count = int(
    (
        pyhmmer_pago_annotated_metadata_dataframe["pyhmmer_evidence_class"]
        == "no_selected_profile_hit"
    ).sum()
)

print("PyHMMER pAgo-related profile annotation is ready.")
print(f"Profile HMM file path: {pyhmmer_pago_result.profile_hmm_file_path}")
print(f"Domain hit count: {len(pyhmmer_pago_domain_hits_dataframe)}")
print(
    "Sequences with any selected Argonaute/PIWI RE profile hit: "
    f"{len(pyhmmer_pago_sequence_summary_dataframe)}"
)
print(
    "Classic pAgo initial candidates with Piwi profile: "
    f"{pyhmmer_classic_piwi_candidate_count}"
)
print(
    "PIWI RE module candidates with MID_pPIWI_RE profile: "
    f"{pyhmmer_ppiwi_re_candidate_count}"
)
print(
    "PIWI RE accessory-only profile hits: "
    f"{pyhmmer_ppiwi_re_accessory_only_count}"
)
print(
    "Isolated non-PIWI classic Ago profile hits requiring review: "
    f"{pyhmmer_isolated_non_piwi_classic_count}"
)
print(f"Sequences without selected profile hits: {pyhmmer_no_selected_profile_count}")
print(f"Domain hits CSV: {pyhmmer_pago_domain_hits_file_path}")
print(f"Sequence summary CSV: {pyhmmer_pago_sequence_summary_file_path}")
print(f"Annotated metadata CSV: {pyhmmer_pago_annotated_metadata_file_path}")

pyhmmer_pago_annotated_metadata_dataframe.head()


PyHMMER pAgo annotation is ready.
Profile HMM file path: C:\Programming\Python\pAgo-project\data\03-features\hmmer_pago\latest\pago_pfam_profiles.hmm
Domain hit count: 18559
Sequences with any selected HMM profile hit: 16053
Sequences with direct pAgo HMM support: 8000
Sequences with only pPIWI_RE-associated HMM support: 8053
Domain hits CSV: C:\Programming\Python\pAgo-project\data\03-features\hmmer_pago\latest\pyhmmer_pago_domain_hits.csv
Sequence summary CSV: C:\Programming\Python\pAgo-project\data\03-features\hmmer_pago\latest\pyhmmer_pago_sequence_summary.csv
Annotated metadata CSV: C:\Programming\Python\pAgo-project\data\03-features\hmmer_pago\latest\sweep_genes_metadata_with_pyhmmer_pago.csv


,sequence_index,record_id,description,sequence_length,pyhmmer_pago_domain_count,pyhmmer_pago_profile_count,pyhmmer_pago_profiles,pyhmmer_pago_profile_accessions,pyhmmer_pago_best_domain_i_evalue,pyhmmer_pago_best_domain_score,pyhmmer_has_piwi,pyhmmer_has_paz,pyhmmer_has_core_argonaute_domain,pyhmmer_has_ppiwi_re_domain,pyhmmer_has_ppiwi_re_associated_domain,pyhmmer_supports_pago
0,0,protein_uid=1000250755|accession=KXK13845.1|le...,protein_uid=1000250755|accession=KXK13845.1|le...,709,0,0,,,NaN,NaN,False,False,False,False,False,False
1,1,protein_uid=1000266463|accession=KXK28958.1|le...,protein_uid=1000266463|accession=KXK28958.1|le...,1043,0,0,,,NaN,NaN,False,False,False,False,False,False
2,2,protein_uid=1000285434|accession=KXK47085.1|le...,protein_uid=1000285434|accession=KXK47085.1|le...,365,0,0,,,NaN,NaN,False,False,False,False,False,False
3,3,protein_uid=1000285973|accession=KXK47585.1|le...,protein_uid=1000285973|accession=KXK47585.1|le...,684,1,1,Piwi,PF02171,9.838423e-15,55.749615,True,False,True,False,False,True
4,4,protein_uid=1000287044|accession=KXK48581.1|le...,protein_uid=1000287044|accession=KXK48581.1|le...,713,0,0,,,NaN,NaN,False,False,False,False,False,False


In [7]:
# =============================================================================
# CELL 7 - Expose downstream variables
# =============================================================================

print("Variables exposed for downstream notebooks:")
print("- protein_fasta_snapshot_directory")
print("- protein_fasta_file_path")
print("- protein_fasta_manifest_file_path")
print("- protein_fasta_manifest_payload")
print("- sweep_genes_snapshot_directory")
print("- sweep_genes_embeddings_file_path")
print("- sweep_genes_manifest_file_path")
print("- sweep_genes_manifest_payload")
print("- sweep_genes_embeddings")
print("- sweep_genes_sequence_metadata_dataframe")
print("- pyhmmer_pago_domain_hits_dataframe")
print("- pyhmmer_pago_sequence_summary_dataframe")
print("- pyhmmer_pago_annotated_metadata_dataframe")
print("- pyhmmer_pago_manifest_payload")
print("- pyhmmer_pago_manifest_file_path")
print("- pyhmmer_pago_domain_hits_file_path")
print("- pyhmmer_pago_sequence_summary_file_path")
print("- pyhmmer_pago_annotated_metadata_file_path")


Variables exposed for downstream notebooks:
- protein_fasta_snapshot_directory
- protein_fasta_file_path
- protein_fasta_manifest_file_path
- protein_fasta_manifest_payload
- sweep_genes_snapshot_directory
- sweep_genes_embeddings_file_path
- sweep_genes_manifest_file_path
- sweep_genes_manifest_payload
- sweep_genes_embeddings
- sweep_genes_sequence_metadata_dataframe
- pyhmmer_pago_domain_hits_dataframe
- pyhmmer_pago_sequence_summary_dataframe
- pyhmmer_pago_annotated_metadata_dataframe
- pyhmmer_pago_manifest_payload
- pyhmmer_pago_manifest_file_path
- pyhmmer_pago_domain_hits_file_path
- pyhmmer_pago_sequence_summary_file_path
- pyhmmer_pago_annotated_metadata_file_path
